# PCA of called genotypes

**Purpose.** Run a PCA the ordinary way — on genotypes that have already been called —
and read the result against the admixture proportions for the same individuals. This is
the situation you are in with array data or high depth sequencing, where the genotypes
can be trusted.

**What you will do**
 - run PCAone on an LD-pruned plink fileset
 - plot the principal components, marking population and super-population separately
 - compare the PCA with the admixture proportions for the same individuals
 - look at the later PCs and ask what they capture

**The data.** 192 individuals from the **1000 Genomes Project**: 16 populations, **12
individuals from each**, grouped into four super-populations.

| Super-population | Populations |
|---|---|
| AFR (Africa) | ACB Barbados, ASW southwestern USA, ESN Esan Nigeria, GWD Gambia, MSL Mende Sierra Leone, YRI Yoruba Nigeria |
| AMR (Americas) | CLM Medellín Colombia, MXL Mexican ancestry Los Angeles, PEL Lima Peru |
| EAS (East Asia) | CHB Han Chinese Beijing, CHS Southern Han Chinese, JPT Japanese Tokyo |
| EUR (Europe) | CEU Utah northern/western European ancestry, GBR Britain, IBS Spain, TSI Italy |

**317,850 autosomal SNPs**, already pruned for linkage disequilibrium, in plink binary
format (`.bed`/`.bim`/`.fam`). The balanced design — the same number of individuals per
population — matters: an unbalanced sample can bend the principal components towards
whichever population is largest.

**Before this** do [MDS and PCA by hand](pca_mds_and_svd.ipynb).

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/current_data/PCA/human_called

# where you will do the exercise
WORK_DIR=$HOME/pca_called_genotypes_human

mkdir -p $WORK_DIR/pca
echo $WORK_DIR > $HOME/.pca_called_human_workdir
cd $WORK_DIR

# link the input files into the working folder
cp -sf $DATA/human_autosomes_12pp_pcaoneLD02* ./pca/

echo --programs that are installed:--
which PCAone

echo; echo --- files in folder ---
ls pca/

In [ ]:
# the working directory was set in the first cell of the notebook
work_d <- readLines(path.expand("~/.pca_called_human_workdir"))[1]
setwd(work_d)
getwd()

# PCA of called genotypes with PCAone

Here the genotypes are already called and LD-pruned, stored as a plink binary
fileset. We run PCAone on them and compare the result with the admixture
proportions for the same individuals.

If you have not done the [MDS and PCA by hand](pca_mds_and_svd.ipynb) exercise,
do that one first.

In [ ]:
# the files were linked into the folder in the setup cell
ls pca/


Look inside the first lines in the population information file. 

In [ ]:
echo --- first 10 lines of the popluation information ---

head ./pca/human_autosomes_12pp_pcaoneLD02.labels.tsv

echo --- summaries the second column ---
cut -d' ' -f2 ./pca/human_autosomes_12pp_pcaoneLD02.labels.tsv | sort |  uniq -c

echo --- summaries the third column ---
cut -d' ' -f3 ./pca/human_autosomes_12pp_pcaoneLD02.labels.tsv | sort |  uniq -c

In [ ]:
from jupyterquiz import display_quiz
display_quiz('pcaone_quiz1.json')


 
 ## Run PCAone to perform PCA
 First let's get a list of the options in PCAone


In [ ]:
${TOOL_PATH}/PCAone

 
 PCAone is a fast and scalable tool for PCA analysis.
 
 For this small dataset, it will be done in **~5 seconds**

In [ ]:
${TOOL_PATH}/PCAone -b ./pca/human_autosomes_12pp_pcaoneLD02 -k 10 -d 0 -o ./pca/PCAONE_K10 -n 6

**Questions**
 - How many individuals and how many SNPs did PCAone use?
 - `-k 10` asks for 10 components. How would you ask for more?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/pca/quiz/pcaone1.json")

Let's look at the output of PCAone. Can you figure out what's stored in these output files?

In [ ]:
ls pca/PCAONE_K10.*

In [ ]:
from jupyterquiz import display_quiz
display_quiz('pcaone_quiz2.json')


Now let's make PCA plots!

In [ ]:
# Read eigenvector matrix calculated by PCAone
e <- read.table("./pca/PCAONE_K10.eigvecs")
# Read population and super-population labels for each individuals
labels<-read.table("./pca/human_autosomes_12pp_pcaoneLD02.labels.tsv",stringsAsFactors=T,head=F)
# use pch for different pop, use color for super-pops
pchs <- 1:20
plot(e[,1:2], pch=as.integer(factor(labels[,2])), col=factor(labels[,3]), xlab="PC1", ylab="PC2", cex=2)
legend("topleft",fill=1:4,levels(labels[,3]))
legend("top",legend=levels(labels[,2]),pch=pchs[seq_along(levels(labels[,2]))],col="black",bty="n",cex=1.2)

**Question**
 - The eigenvector file has one row per individual. Which column is PC1?

Compare with the estimate admixture proportions from ADMIXTURE analysis this morning



In [ ]:
#read in code to plot admixture proportions ( plotAdmix function)
#source("https://raw.githubusercontent.com/GenisGE/evalAdmix/master/visFuns.R")

options(repr.plot.width=12, repr.plot.height=12)
layout(matrix(c(1,1,2,3),nrow=2,by=T),height=c(2,4),width=2:1)
# Read population and super-population labels for each individuals
labels<-read.table("./pca/human_autosomes_12pp_pcaoneLD02.labels.tsv",stringsAsFactors=T,head=F)
pop <- labels[,2]
pop_order <- c("GWD", "MSL", "YRI", "ESN", "ACB", "ASW",
               "GBR", "CEU", "IBS", "TSI",
               "CHB", "CHS", "JPT",
               "MXL", "CLM", "PEL")
ord <- unlist(lapply(pop_order, function(p) which(pop == p)))

admix_cols <- c(AMR="#FF7F00", AFR="#4DAF4A", EUR="#377EB8", EAS="#984EA3")

reorder_components <- function(q) {
  ref <- c("PEL", "YRI", "GBR", "CHB")
  kord <- sapply(ref, function(p) which.max(colMeans(q[pop == p,,drop=FALSE])))
  if(length(unique(kord)) == ncol(q)) q[,kord] else q
}

plot_geo_admix <- function(q, title) {
  plotAdmix(q, pop=pop, ord=ord, rotatelab=45, padj=0.12,
            cex.lab=1.0, cex.main=1.3, main=title,
            colorpal=unname(admix_cols[c("AMR", "AFR", "EUR", "EAS")]))
}
                 
# Read in inferred admixture proportions
q_best <- read.table("./pca/human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q")
q_best <- reorder_components(as.matrix(q_best))
plot_geo_admix(q_best, "ADMIXTURE proportions, K = 4, best seed = 5")
# Read eigenvector matrix calculated by PCAone
e <- read.table("./pca/PCAONE_K10.eigvecs")

pchs <- 1:20
# use pch for different pop, use color for super-pops
plot(e[,1:2], pch=pchs[factor(labels[,2])], col=factor(labels[,3]), xlab="PC1", ylab="PC2", cex=2)
legend("topleft",fill=1:4,levels(labels[,3]))
plot.new()
par(mar=c(0,0,0,0))
legend("top",legend=levels(labels[,2]),pch=pchs[seq_along(levels(labels[,2]))],col="black",bty="n",cex=1.5)

**Questions**
 - Compare the PCA with the admixture proportions. Which individuals look admixed in both?
 - What does the PCA show that the admixture bar plot does not?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/pca/quiz/pcaone2.json")

 **Questions**
 - What information do you get from the PCA that you don't get from the ADMIXTURE results?
 - Can you identify the admixed individuals?
 
 Lets see what the other PCs show. 
 

In [ ]:
par(mfrow=c(3,2))
for(pc in 1:5)
  plot(e[,1:2+2*(pc-1)], pch=pchs[as.integer(factor(labels[,2]))], col=factor(labels[,3]), ylab=paste("PC",pc*2),xlab=paste("PC",pc*2-1), cex=2);

**Questions**
 - How many PCs are needed to separate the populations?
 - What do you think is captured by the later PCs?

 **Questions**
 - How many PCs are used to separate the populations?
 - What do you think is captured on PC 5 and 6?
 - What do you think is captured on PC 9 and 10? 